# TinyLlama Oracle LoRA — **V15 Audit AI MAX4000 torchao clean**


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation propre V15 MAX4000                ║
# ║  Fix définitif : suppression de torchao incompatible Colab  ║
# ║  Exécuter cette cellule seule, puis laisser Colab redémarrer ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, os

def run(cmd, check=False):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

# 1) Supprimer torchao fourni par Colab.
# PEFT fonctionne sans torchao, mais échoue si torchao 0.10.0 est installé.
run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

# 2) Installer les dépendances utiles sans réinstaller torchao.
run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
     "numpy>=2.0.0,<2.1.0",
     "transformers>=4.40.0",
     "peft>=0.10.0",
     "accelerate>=0.28.0",
     "datasets>=2.18.0",
     "bitsandbytes>=0.43.0",
     "huggingface_hub>=0.22.0",
     "pandas>=2.0.0",
     "torch>=2.2.0"], check=True)

# 3) Sécurité : si torchao est revenu malgré tout, le retirer une deuxième fois.
run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

# 4) Vérification avant redémarrage.
try:
    import importlib.metadata as md
    print("torchao détecté :", md.version("torchao"))
    print("⚠️ torchao est encore présent. Redémarrage nécessaire après désinstallation.")
except Exception:
    print("✅ torchao absent : PEFT ne déclenchera plus l'erreur torchao 0.10.0.")

print("✅ Installation terminée. Redémarrage automatique du kernel...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)



In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Vérification GPU + versions                     ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, numpy as np, torch, transformers, peft, accelerate, datasets
try:
    import torchao
    TORCHAO_VERSION = torchao.__version__
except Exception:
    TORCHAO_VERSION = "non installé"

print("=" * 70)
print("  ENVIRONNEMENT COLAB — V15")
print("=" * 70)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  NumPy        : {np.__version__}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  PEFT         : {peft.__version__}")
print(f"  Accelerate   : {accelerate.__version__}")
print(f"  Datasets     : {datasets.__version__}")
print("=" * 70)

if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    total_gb = dev.total_memory / 1e9
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"  GPU          : {dev.name}")
    print(f"  VRAM totale  : {total_gb:.1f} GB")
    print(f"  VRAM libre   : {free_gb:.1f} GB")
    print("  ✅ GPU disponible")
else:
    print("  ⚠️ Pas de GPU — entraînement impossible")
print("=" * 70)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Téléchargement TinyLlama-1.1B-Chat-v1.0         ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import snapshot_download
import shutil, os

MODEL_DIR = "TinyLlama-1.1B-Chat-v1.0"

if not os.path.exists(MODEL_DIR):
    print("📥 Téléchargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...")
    snapshot_download(
        repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )
    print("✅ Modèle téléchargé.")
else:
    print(f"✅ Modèle déjà présent : {MODEL_DIR}")

if not os.path.exists(MODEL_DIR + ".zip"):
    shutil.make_archive(MODEL_DIR, "zip", MODEL_DIR)
    print(f"📦 Archive : {MODEL_DIR}.zip")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Table d'audit simulée orientée production V15   ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd
from datetime import datetime, timedelta

random.seed(42)

DB_USERS = [
    "SYS", "SYSTEM", "CYRILLE", "CYRILLE_TBS", "ITEST", "HR",
    "SMART2DADMIN", "SMART2DADMINI", "BATCH_USER", "PROD2_MDS",
    "PROD2_STB", "AUDIT_MANAGER", "REPORT_USER", "NEW_USER_APP"
]

OBJECTS = [
    "EMPLOYEES", "DEPARTMENTS", "HR", "ADRESS", "CLIENT",
    "FACTURES", "COMMANDES", "PAIEMENTS", "CONTRATS",
    "AUD$", "VROMUALD", "TEST", "TEST_ROMUALD", "DEVICE_ADDRESS",
    "SCHEMA_VERSION_REGISTRY"
]

ACTIONS = [
    "LOGON", "LOGOFF", "SELECT", "INSERT", "UPDATE", "DELETE",
    "GRANT", "REVOKE", "ALTER", "CREATE USER", "DROP USER", "ALTER USER"
]

HOSTS = ["srv-oracle-01", "app-server-01", "poste-rh-07", "laptop-cyrille", "srv-batch-02"]
CLIENTS = ["sqlplus", "TOAD", "JDBC", "Python", "PowerBI", "AuditAI"]

base_dt = datetime(2025, 1, 1)
rows = []

for i in range(2500):
    dt = base_dt + timedelta(
        days=random.randint(0, 520),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59),
    )
    action = random.choice(ACTIONS)
    obj = None if action in ("LOGON", "LOGOFF") else random.choice(OBJECTS)
    rows.append({
        "ID": i + 1,
        "AUDIT_TYPE": random.choice(["STANDARD", "UNIFIED"]),
        "SESSIONID": random.randint(100000, 999999),
        "OS_USERNAME": random.choice(["oracle", "system", "cyrille", "smart2dadmin", "batch"]),
        "USERHOST": random.choice(HOSTS),
        "TERMINAL": f"pts/{random.randint(0, 9)}",
        "AUTHENTICATION_TYPE": random.choice(["PASSWORD", "KERBEROS", "CERTIFICATE", "NONE"]),
        "DBUSERNAME": random.choice(DB_USERS),
        "CLIENT_PROGRAM_NAME": random.choice(CLIENTS),
        "OBJECT_SCHEMA": random.choice(["HR", "SYS", "SMART2DSECU", "APP_SCHEMA"]),
        "OBJECT_NAME": obj,
        "SQL_TEXT": f"/* simulation */ {action} ON {obj}" if obj else None,
        "SQL_BINDS": None,
        "EVENT_TIMESTAMP": dt.strftime("%Y-%m-%dT%H:%M:%S.%f"),
        "ACTION_NAME": action,
        "RETURNCODE": 0 if random.random() > 0.08 else random.choice([1017, 1031, 904]),
        "INSTANCE": 1,
    })

AUDIT_DF = pd.DataFrame(rows)
AUDIT_DF.to_csv("oracle_audit_trail.csv", index=False)

print(f"✅ oracle_audit_trail.csv généré : {len(AUDIT_DF)} lignes")
print(AUDIT_DF[["ID", "DBUSERNAME", "ACTION_NAME", "OBJECT_NAME", "EVENT_TIMESTAMP"]].head(10).to_string(index=False))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Référence colonnes / valeurs métier             ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv")

print("╔════════════════════════════════════════════════════════════════╗")
print("║      ORACLE_AUDIT_TRAIL — RÉFÉRENCE V15                       ║")
print("╠════════════════════════════════════════════════════════════════╣")
print(f"║  Lignes : {len(AUDIT_DF):<6} Colonnes : {len(AUDIT_DF.columns):<4}                              ║")
print("╚════════════════════════════════════════════════════════════════╝")

print("\nColonnes :")
for col in AUDIT_DF.columns:
    sample = AUDIT_DF[col].dropna()
    val = str(sample.iloc[0])[:45] if not sample.empty else "NULL"
    print(f"  - {col:<24} ex: {val}")

print("\nUtilisateurs observés :", sorted(AUDIT_DF["DBUSERNAME"].dropna().unique()))
print("Objets observés       :", sorted(AUDIT_DF["OBJECT_NAME"].dropna().unique()))
print("Actions observées     :", sorted(AUDIT_DF["ACTION_NAME"].dropna().unique()))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Configuration V15 + générateurs SQL             ║
# ╚══════════════════════════════════════════════════════════════╝
import random, re, pandas as pd
from collections import defaultdict

random.seed(42)

TABLE = "ORACLE_AUDIT_TRAIL"
TS = "EVENT_TIMESTAMP"
USER_C = "DBUSERNAME"
OBJ_C = "OBJECT_NAME"
ACT_C = "ACTION_NAME"
HOST_C = "USERHOST"
TERM_C = "TERMINAL"
OS_USER_C = "OS_USERNAME"
CLIENT_C = "CLIENT_PROGRAM_NAME"
RET_C = "RETURNCODE"

CANONICAL_USERS = [
    "SYSTEM", "SYS", "CYRILLE", "CYRILLE_TBS", "ITEST", "HR",
    "VROMUALD", "SMART2DADMIN", "BATCH_USER", "PROD2_MDS", "PROD2_STB",
    "AUDIT_MANAGER", "REPORT_USER", "NEW_USER_APP"
]

CANONICAL_OBJECTS = [
    "EMPLOYEES", "DEPARTMENTS", "HR", "ADRESS", "CLIENT",
    "FACTURES", "COMMANDES", "PAIEMENTS", "CONTRATS",
    "SALARIES", "FOURNISSEURS", "DOSSIERS", "UTILISATEURS",
    "NOUVELLE_TABLE_METIER"
]

ACTIONS = ["SELECT", "INSERT", "UPDATE", "DELETE", "GRANT", "REVOKE", "ALTER", "LOGON", "LOGOFF", "CREATE USER", "DROP USER", "ALTER USER"]

ACTION_SYNONYMS = {
    "SELECT": ["SELECT", "consulté", "lu", "regardé", "accédé à", "ouvert", "affiché"],
    "INSERT": ["INSERT", "inséré", "ajouté", "créé une ligne dans", "ajouté des données dans"],
    "UPDATE": ["UPDATE", "modifié", "mis à jour", "changé", "corrigé"],
    "DELETE": ["DELETE", "supprimé", "effacé", "retiré des lignes de"],
    "GRANT": ["GRANT", "donné un droit", "accordé un privilège", "autorisé"],
    "REVOKE": ["REVOKE", "retiré un droit", "révoqué un privilège"],
    "ALTER": ["ALTER", "modifié la structure de", "changé la structure de"],
    "LOGON": ["connecté", "connexion", "login", "ouvert une session"],
    "LOGOFF": ["déconnecté", "déconnexion", "fermé une session"],
    "CREATE USER": ["créé un utilisateur", "créé un compte", "création de compte"],
    "DROP USER": ["supprimé un utilisateur", "supprimé un compte"],
    "ALTER USER": ["modifié un utilisateur", "changé un compte"],
}

GENERIC_PREFIXES = [
    "", "Question : ", "Audit : ", "Analyse : ", "Contrôle : ", "Sécurité : ",
    "Peux-tu me dire ", "Je veux savoir ", "Montre-moi ", "Vérifie "
]

def norm_value(v: str) -> str:
    return str(v).strip().replace("'", "''").upper()

def ci_eq(column: str, value: str) -> str:
    return f"UPPER({column})='{norm_value(value)}'"

def ci_in(column: str, values) -> str:
    vals = ", ".join(f"'{norm_value(v)}'" for v in values)
    return f"UPPER({column}) IN ({vals})"

def not_null(column: str) -> str:
    return f"{column} IS NOT NULL"

def where(conditions) -> str:
    conditions = [c for c in conditions if c]
    return (" WHERE " + " AND ".join(conditions)) if conditions else ""

def select_detail(conditions=None, cols=None, limit=100, order=True) -> str:
    cols = cols or f"{USER_C}, {ACT_C}, {OBJ_C}, {TS}"
    sql = f"SELECT {cols} FROM {TABLE}{where(conditions or [])}"
    if order:
        sql += f" ORDER BY {TS} DESC"
    if limit:
        sql += f" FETCH FIRST {limit} ROWS ONLY"
    return sql + ";"

def select_count(conditions=None, alias="NB") -> str:
    return f"SELECT COUNT(*) AS {alias} FROM {TABLE}{where(conditions or [])};"

def select_exists(conditions=None) -> str:
    return select_count(conditions, alias="NB_ACTIONS")

def select_group(group_col, conditions=None, alias="NB", limit=None) -> str:
    sql = f"SELECT {group_col}, COUNT(*) AS {alias} FROM {TABLE}{where(conditions or [])} GROUP BY {group_col} ORDER BY {alias} DESC"
    if limit:
        sql += f" FETCH FIRST {limit} ROWS ONLY"
    return sql + ";"

def select_distinct(col, conditions=None) -> str:
    return f"SELECT DISTINCT {col} FROM {TABLE}{where(conditions or [])} ORDER BY {col};"

def date_range(label: str):
    label = label.lower().strip()
    # Les conditions utilisent des bornes pour éviter TRUNC(colonne)=... quand possible.
    mapping = {
        "aujourd'hui": f"{TS} >= TRUNC(SYSDATE) AND {TS} < TRUNC(SYSDATE+1)",
        "aujourd hui": f"{TS} >= TRUNC(SYSDATE) AND {TS} < TRUNC(SYSDATE+1)",
        "hier": f"{TS} >= TRUNC(SYSDATE-1) AND {TS} < TRUNC(SYSDATE)",
        "avant-hier": f"{TS} >= TRUNC(SYSDATE-2) AND {TS} < TRUNC(SYSDATE-1)",
        "avant hier": f"{TS} >= TRUNC(SYSDATE-2) AND {TS} < TRUNC(SYSDATE-1)",
        "ce mois": f"{TS} >= TRUNC(SYSDATE,'MM') AND {TS} < ADD_MONTHS(TRUNC(SYSDATE,'MM'),1)",
        "mois dernier": f"{TS} >= ADD_MONTHS(TRUNC(SYSDATE,'MM'),-1) AND {TS} < TRUNC(SYSDATE,'MM')",
        "ce mois-ci": f"{TS} >= TRUNC(SYSDATE,'MM') AND {TS} < ADD_MONTHS(TRUNC(SYSDATE,'MM'),1)",
        "cette semaine": f"{TS} >= TRUNC(SYSDATE,'IW') AND {TS} < TRUNC(SYSDATE,'IW')+7",
        "semaine dernière": f"{TS} >= TRUNC(SYSDATE,'IW')-7 AND {TS} < TRUNC(SYSDATE,'IW')",
        "la nuit": f"TO_NUMBER(TO_CHAR({TS},'HH24')) BETWEEN 0 AND 5",
        "dans la nuit": f"TO_NUMBER(TO_CHAR({TS},'HH24')) BETWEEN 0 AND 5",
        "nuit dernière": f"{TS} >= TRUNC(SYSDATE-1)+20/24 AND {TS} < TRUNC(SYSDATE)+6/24",
        "la nuit dernière": f"{TS} >= TRUNC(SYSDATE-1)+20/24 AND {TS} < TRUNC(SYSDATE)+6/24",
    }
    return mapping[label]

def last_n_days(n: int) -> str:
    return f"{TS} >= SYSDATE-{int(n)}"

def exact_n_days_ago(n: int) -> str:
    n = int(n)
    return f"{TS} >= TRUNC(SYSDATE-{n}) AND {TS} < TRUNC(SYSDATE-{n-1})"

def n_months_ago(n: int) -> str:
    n = int(n)
    return f"{TS} >= ADD_MONTHS(TRUNC(SYSDATE,'MM'),-{n}) AND {TS} < ADD_MONTHS(TRUNC(SYSDATE,'MM'),-{n-1})"

def weekday_last(day_name: str) -> str:
    offsets = {
        "lundi": 0, "mardi": 1, "mercredi": 2, "jeudi": 3,
        "vendredi": 4, "samedi": 5, "dimanche": 6
    }
    off = offsets[day_name.lower()]
    start = f"TRUNC(SYSDATE,'IW')-7+{off}"
    return f"{TS} >= {start} AND {TS} < {start}+1"

def add(examples, instruction, output, source):
    q = re.sub(r"\s+", " ", instruction).strip()
    sql = re.sub(r"\s+", " ", output).strip()
    if not sql.endswith(";"):
        sql += ";"
    examples.append({"instruction": q, "output": sql, "source": source})

def add_with_prefixes(examples, question, sql, source, prefixes=None):
    prefixes = prefixes if prefixes is not None else GENERIC_PREFIXES
    for pfx in prefixes:
        q = (pfx + question).strip()
        add(examples, q, sql, source)

def sample_many(items, k):
    return [random.choice(items) for _ in range(k)]


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Bloc A : intentions atomiques solides           ║
# ╚══════════════════════════════════════════════════════════════╝
blocA = []

# A1 — Listes utiles pour les colonnes d'aide
base_pairs = [
    ("Quels utilisateurs apparaissent dans les traces d'audit ?", select_distinct(USER_C, [not_null(USER_C)])),
    ("Liste les utilisateurs présents dans l'audit.", select_distinct(USER_C, [not_null(USER_C)])),
    ("Combien d'utilisateurs distincts apparaissent dans les traces ?", f"SELECT COUNT(DISTINCT {USER_C}) AS NB_UTILISATEURS FROM {TABLE} WHERE {USER_C} IS NOT NULL;"),
    ("Quelles tables sont mentionnées dans l'audit ?", select_distinct(OBJ_C, [not_null(OBJ_C)])),
    ("Liste les tables touchées dans les journaux.", select_distinct(OBJ_C, [not_null(OBJ_C)])),
    ("Combien de tables différentes ont été touchées ?", f"SELECT COUNT(DISTINCT {OBJ_C}) AS NB_TABLES FROM {TABLE} WHERE {OBJ_C} IS NOT NULL;"),
    ("Quelles actions existent dans l'audit ?", select_distinct(ACT_C, [not_null(ACT_C)])),
    ("Liste les types d'actions auditées.", select_distinct(ACT_C, [not_null(ACT_C)])),
]
for q, sql in base_pairs:
    add_with_prefixes(blocA, q, sql, "A_lists")

# A2 — Dernières actions globales, utilisateur, table, action
for u in CANONICAL_USERS:
    sql = select_detail([ci_eq(USER_C, u)], limit=1)
    for q in [
        f"Quelle est la dernière action effectuée par l'utilisateur {u} ?",
        f"Dernière action de {u}.",
        f"Qu'a fait {u} en dernier dans la base ?",
        f"Montre la dernière activité du compte {u}.",
        f"Le dernier événement de {u}, c'est quoi ?",
    ]:
        add_with_prefixes(blocA, q, sql, "A_last_user", prefixes=["", "Question : ", "Audit : "])

for obj in CANONICAL_OBJECTS:
    sql = select_detail([ci_eq(OBJ_C, obj)], limit=1)
    for q in [
        f"Quelle est la dernière action sur la table {obj} ?",
        f"Dernier événement concernant {obj}.",
        f"Qui a touché {obj} en dernier ?",
        f"Qu'est-ce qui s'est passé en dernier sur {obj} ?",
    ]:
        add_with_prefixes(blocA, q, sql, "A_last_object", prefixes=["", "Question : ", "Analyse : "])

for act in ["SELECT", "INSERT", "UPDATE", "DELETE", "GRANT", "REVOKE", "ALTER", "LOGON", "LOGOFF"]:
    sql = select_detail([ci_eq(ACT_C, act)], limit=1)
    for q in [
        f"Quelle est la dernière action {act} ?",
        f"Montre le dernier {act}.",
        f"Dernier événement de type {act}.",
    ]:
        add_with_prefixes(blocA, q, sql, "A_last_action", prefixes=["", "Audit : "])

# A3 — Global dernière action dans la base : ne pas ajouter de filtre OBJECT_NAME
global_last_sql = select_detail([], limit=1)
for q in [
    "Quelle est la dernière action effectuée dans la base ?",
    "Quelle est la dernière action enregistrée ?",
    "Qu'est-ce qui s'est passé en dernier dans la base ?",
    "Dernier événement d'audit.",
    "Montre-moi la toute dernière trace.",
]:
    add_with_prefixes(blocA, q, global_last_sql, "A_last_global", prefixes=["", "Question : ", "Audit : "])

# A4 — Classements
rank_pairs = [
    ("Quel utilisateur a fait le plus d'actions ?", select_group(USER_C, [not_null(USER_C)], limit=1)),
    ("Top 5 des utilisateurs les plus actifs.", select_group(USER_C, [not_null(USER_C)], limit=5)),
    ("Classe les utilisateurs par activité.", select_group(USER_C, [not_null(USER_C)])),
    ("Quelle table a été la plus touchée ?", select_group(OBJ_C, [not_null(OBJ_C)], limit=1)),
    ("Top 10 des tables les plus touchées.", select_group(OBJ_C, [not_null(OBJ_C)], limit=10)),
    ("Quelles actions sont les plus fréquentes ?", select_group(ACT_C, [not_null(ACT_C)])),
]
for q, sql in rank_pairs:
    add_with_prefixes(blocA, q, sql, "A_rankings")

print(f"Bloc A : {len(blocA)} exemples")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Bloc B : utilisateur + table + action           ║
# ╚══════════════════════════════════════════════════════════════╝
blocB = []

# B1 — Action sur table : qui ?
for obj in CANONICAL_OBJECTS:
    for act in ["SELECT", "INSERT", "UPDATE", "DELETE", "GRANT", "REVOKE", "ALTER"]:
        cond = [ci_eq(OBJ_C, obj), ci_eq(ACT_C, act), not_null(USER_C)]
        sql_users = select_group(USER_C, cond)
        synonyms = ACTION_SYNONYMS.get(act, [act])
        for syn in synonyms[:4]:
            questions = [
                f"Qui a fait un {act} sur la table {obj} ?",
                f"Qui a {syn} {obj} ?",
                f"Quels utilisateurs ont réalisé {act} sur {obj} ?",
                f"Je veux savoir qui a fait {act} sur {obj}.",
            ]
            for q in questions:
                add_with_prefixes(blocB, q, sql_users, "B_who_action_object", prefixes=["", "Question : "])

# B2 — Utilisateur + table : détail et existence
for u in CANONICAL_USERS:
    for obj in CANONICAL_OBJECTS:
        cond = [ci_eq(USER_C, u), ci_eq(OBJ_C, obj)]
        sql_detail = select_detail(cond, limit=100)
        sql_count = select_exists(cond)
        for q in [
            f"Est-ce que {u} a touché la table {obj} ?",
            f"{u} a-t-il fait une action sur {obj} ?",
            f"Y a-t-il une activité de {u} sur {obj} ?",
            f"Montre ce que {u} a fait sur {obj}.",
            f"Donne les actions de {u} concernant {obj}.",
        ]:
            sql = sql_count if q.lower().startswith(("est-ce", f"{u.lower()} a-t-il", "y a-t-il")) else sql_detail
            add_with_prefixes(blocB, q, sql, "B_user_object", prefixes=["", "Audit : "])

# B3 — Utilisateur + action
for u in CANONICAL_USERS:
    for act in ["SELECT", "INSERT", "UPDATE", "DELETE", "GRANT", "REVOKE", "ALTER", "LOGON", "LOGOFF"]:
        cond = [ci_eq(USER_C, u), ci_eq(ACT_C, act)]
        detail = select_detail(cond, limit=100)
        count = select_count(cond, alias="NB_ACTIONS")
        for q in [
            f"Quelles actions {act} ont été faites par {u} ?",
            f"Montre les {act} de {u}.",
            f"Combien de {act} pour {u} ?",
            f"Est-ce que {u} a fait un {act} ?",
        ]:
            sql = count if q.lower().startswith(("combien", "est-ce")) else detail
            add_with_prefixes(blocB, q, sql, "B_user_action", prefixes=["", "Question : "])

# B4 — Utilisateur + action + table : très important contre les oublis de filtres
for u in CANONICAL_USERS:
    for obj in CANONICAL_OBJECTS:
        for act in ["SELECT", "INSERT", "UPDATE", "DELETE"]:
            cond = [ci_eq(USER_C, u), ci_eq(ACT_C, act), ci_eq(OBJ_C, obj)]
            detail = select_detail(cond, limit=100)
            count = select_count(cond, alias="NB_ACTIONS")
            for q in [
                f"Est-ce que l'utilisateur {u} a fait un {act} sur la table {obj} ?",
                f"{u} a-t-il fait {act} sur {obj} ?",
                f"Montre les {act} de {u} sur {obj}.",
                f"Donne les traces où {u} fait {act} sur {obj}.",
            ]:
                sql = count if q.lower().startswith(("est-ce", f"{u.lower()} a-t-il")) else detail
                add(blocB, q, sql, "B_user_action_object")

# B5 — Actions métier génériques
modified_cond = [ci_in(ACT_C, ["INSERT", "UPDATE", "DELETE"])]
for obj in CANONICAL_OBJECTS:
    cond = modified_cond + [ci_eq(OBJ_C, obj)]
    for q in [
        f"Qui a modifié des données dans {obj} ?",
        f"Qui a changé la table {obj} ?",
        f"Y a-t-il eu des modifications sur {obj} ?",
        f"Quelles modifications ont eu lieu sur {obj} ?",
    ]:
        sql = select_count(cond, "NB_MODIFICATIONS") if q.lower().startswith(("y a-t-il", "est-ce")) else select_detail(cond, limit=100)
        add_with_prefixes(blocB, q, sql, "B_business_modify", prefixes=["", "Sécurité : "])

print(f"Bloc B : {len(blocB)} exemples")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — Bloc C : périodes, dates relatives et nuit      ║
# ╚══════════════════════════════════════════════════════════════╝
blocC = []

# C1 — périodes simples
periods = [
    ("aujourd'hui", date_range("aujourd'hui")),
    ("hier", date_range("hier")),
    ("avant-hier", date_range("avant-hier")),
    ("ce mois", date_range("ce mois")),
    ("mois dernier", date_range("mois dernier")),
    ("cette semaine", date_range("cette semaine")),
    ("semaine dernière", date_range("semaine dernière")),
    ("dans la nuit", date_range("dans la nuit")),
    ("la nuit dernière", date_range("la nuit dernière")),
]

for label, cond_date in periods:
    add_with_prefixes(blocC, f"Qu'est-ce qui s'est passé {label} ?", select_detail([cond_date], limit=100), "C_period_global")
    add_with_prefixes(blocC, f"Qui a touché une table {label} ?", select_detail([not_null(OBJ_C), cond_date], limit=100), "C_period_table")
    add_with_prefixes(blocC, f"Quelles tables ont été touchées {label} ?", select_group(OBJ_C, [not_null(OBJ_C), cond_date]), "C_period_group_tables")
    add_with_prefixes(blocC, f"Quels utilisateurs ont été actifs {label} ?", select_group(USER_C, [not_null(USER_C), cond_date]), "C_period_group_users")
    add_with_prefixes(blocC, f"Combien d'actions {label} ?", select_count([cond_date], "NB_ACTIONS"), "C_period_count")

# C2 — périodes + utilisateur / table / action
for label, cond_date in periods:
    for u in random.sample(CANONICAL_USERS, min(8, len(CANONICAL_USERS))):
        add(blocC, f"Quelle est la dernière action de {u} {label} ?", select_detail([ci_eq(USER_C, u), cond_date], limit=1), "C_user_period_last")
        add(blocC, f"Montre l'activité de {u} {label}.", select_detail([ci_eq(USER_C, u), cond_date], limit=100), "C_user_period_detail")
    for obj in random.sample(CANONICAL_OBJECTS, min(8, len(CANONICAL_OBJECTS))):
        add(blocC, f"Qui a touché {obj} {label} ?", select_detail([ci_eq(OBJ_C, obj), cond_date], limit=100), "C_object_period_detail")
        add(blocC, f"Combien d'actions sur {obj} {label} ?", select_count([ci_eq(OBJ_C, obj), cond_date], "NB_ACTIONS"), "C_object_period_count")
    for act in ["SELECT", "INSERT", "UPDATE", "DELETE", "LOGON", "GRANT"]:
        add(blocC, f"Quels {act} ont eu lieu {label} ?", select_detail([ci_eq(ACT_C, act), cond_date], limit=100), "C_action_period_detail")

# C3 — derniers N jours
for n in [1, 2, 3, 5, 7, 10, 14, 30, 45, 60, 90, 180, 365]:
    cond = last_n_days(n)
    for phr in [f"les {n} derniers jours", f"sur les {n} derniers jours", f"depuis {n} jours"]:
        add_with_prefixes(blocC, f"Qu'est-ce qui s'est passé {phr} ?", select_detail([cond], limit=100), "C_last_n_days")
        add_with_prefixes(blocC, f"Combien d'actions {phr} ?", select_count([cond], "NB_ACTIONS"), "C_last_n_days_count")
        for u in random.sample(CANONICAL_USERS, 4):
            add(blocC, f"Qu'a fait {u} {phr} ?", select_detail([ci_eq(USER_C, u), cond], limit=100), "C_user_last_n_days")
        for obj in random.sample(CANONICAL_OBJECTS, 4):
            add(blocC, f"Qui a touché {obj} {phr} ?", select_detail([ci_eq(OBJ_C, obj), cond], limit=100), "C_object_last_n_days")

# C4 — il y a N jours / N mois
for n in [2, 3, 4, 5, 7, 10, 15, 30]:
    cond = exact_n_days_ago(n)
    for q in [
        f"Qu'est-ce qui s'est passé il y a {n} jours ?",
        f"Qui a fait une action il y a {n} jours ?",
        f"Montre les traces d'il y a {n} jours.",
    ]:
        add_with_prefixes(blocC, q, select_detail([cond], limit=100), "C_exact_day_ago")

for n in [2, 3, 4, 6, 12]:
    cond = n_months_ago(n)
    for q in [
        f"Qu'est-ce qui s'est passé il y a {n} mois ?",
        f"Actions il y a {n} mois.",
        f"Qui a touché une table il y a {n} mois ?",
    ]:
        add_with_prefixes(blocC, q, select_detail([cond], limit=100), "C_months_ago")

# C5 — jours nommés
for day in ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]:
    cond = weekday_last(day)
    for q in [
        f"Qu'est-ce qui s'est passé {day} dernier ?",
        f"Qui a touché une table {day} passé ?",
        f"Montre les actions du {day} dernier.",
    ]:
        add_with_prefixes(blocC, q, select_detail([cond], limit=100), "C_weekday_last")

print(f"Bloc C : {len(blocC)} exemples")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — Bloc D : langage non technique + cas difficiles║
# ╚══════════════════════════════════════════════════════════════╝
blocD = []

# D1 — Langage métier non technique
non_tech = [
    ("Qui a regardé les employés hier ?", select_detail([ci_eq(ACT_C, "SELECT"), ci_eq(OBJ_C, "EMPLOYEES"), date_range("hier")], limit=100)),
    ("Quelqu'un a touché les employés hier ?", select_count([ci_eq(OBJ_C, "EMPLOYEES"), date_range("hier")], "NB_ACTIONS")),
    ("Est-ce que SYSTEM a touché CLIENT ?", select_count([ci_eq(USER_C, "SYSTEM"), ci_eq(OBJ_C, "CLIENT")], "NB_ACTIONS")),
    ("Je veux savoir si SYS a regardé DEPARTMENTS.", select_count([ci_eq(USER_C, "SYS"), ci_eq(ACT_C, "SELECT"), ci_eq(OBJ_C, "DEPARTMENTS")], "NB_ACTIONS")),
    ("Qu'est-ce que CYRILLE_TBS a fait dans la nuit ?", select_detail([ci_eq(USER_C, "CYRILLE_TBS"), date_range("dans la nuit")], limit=100)),
    ("Qui a bidouillé la table ADRESS ?", select_detail([ci_eq(OBJ_C, "ADRESS")], limit=100)),
    ("Montre-moi les choses suspectes dans la nuit.", select_detail([date_range("dans la nuit")], limit=100)),
    ("Qui a donné des droits le mois dernier ?", select_detail([ci_eq(ACT_C, "GRANT"), date_range("mois dernier")], limit=100)),
    ("Qui a retiré des droits ?", select_detail([ci_eq(ACT_C, "REVOKE")], limit=100)),
    ("Quel compte a été créé en dernier ?", select_detail([ci_eq(ACT_C, "CREATE USER")], limit=1)),
    ("Quel compte a été supprimé en dernier ?", select_detail([ci_eq(ACT_C, "DROP USER")], limit=1)),
    ("Qui s'est connecté avant-hier ?", select_detail([ci_eq(ACT_C, "LOGON"), date_range("avant-hier")], cols=f"{USER_C}, {TS}", limit=100)),
    ("Combien de connexions hier ?", select_count([ci_eq(ACT_C, "LOGON"), date_range("hier")], "NB_CONNEXIONS")),
    ("Depuis quelle machine SYSTEM est passé ?", select_group(HOST_C, [ci_eq(USER_C, "SYSTEM"), not_null(HOST_C)])),
    ("Quel programme a été utilisé par SYS ?", select_group(CLIENT_C, [ci_eq(USER_C, "SYS"), not_null(CLIENT_C)])),
    ("Quelles erreurs Oracle ont eu lieu ?", select_detail([f"{RET_C} <> 0"], cols=f"{USER_C}, {ACT_C}, {OBJ_C}, {RET_C}, {TS}", limit=100)),
    ("Qui a eu une erreur de connexion ?", select_detail([ci_eq(ACT_C, "LOGON"), f"{RET_C} <> 0"], cols=f"{USER_C}, {RET_C}, {TS}", limit=100)),
]
for q, sql in non_tech:
    add_with_prefixes(blocD, q, sql, "D_non_technical", prefixes=["", "Question : ", "Audit : ", "Contrôle : "])

# D2 — Cas anti-confusion : SYSTEM/SYS/HR peuvent être utilisateurs, pas objets si la question dit utilisateur/compte
for u in ["SYSTEM", "SYS", "HR"]:
    add_with_prefixes(
        blocD,
        f"Quelle est la dernière action effectuée dans la base par l'utilisateur {u} ?",
        select_detail([ci_eq(USER_C, u)], limit=1),
        "D_anti_user_object_confusion",
        prefixes=["", "Question : ", "Analyse : "],
    )
    add_with_prefixes(
        blocD,
        f"Le compte {u} a-t-il touché la base hier ?",
        select_count([ci_eq(USER_C, u), date_range("hier")], "NB_ACTIONS"),
        "D_anti_user_object_confusion",
        prefixes=["", "Vérifie : "],
    )

# D3 — Cas inverse : HR peut être une table/objet si la question dit table/objet
add_with_prefixes(
    blocD,
    "Qui a touché la table HR ?",
    select_detail([ci_eq(OBJ_C, "HR")], limit=100),
    "D_anti_user_object_confusion",
)
add_with_prefixes(
    blocD,
    "Quelle est la dernière action sur l'objet HR ?",
    select_detail([ci_eq(OBJ_C, "HR")], limit=1),
    "D_anti_user_object_confusion",
)

# D4 — Colonnes USERHOST / TERMINAL / OS_USERNAME / CLIENT_PROGRAM_NAME
for host in ["srv-oracle-01", "app-server-01", "poste-rh-07", "laptop-cyrille"]:
    add_with_prefixes(blocD, f"Quelles actions viennent du poste {host} ?", select_detail([ci_eq(HOST_C, host)], limit=100), "D_host")
    add_with_prefixes(blocD, f"Qui s'est connecté depuis {host} ?", select_detail([ci_eq(HOST_C, host), ci_eq(ACT_C, "LOGON")], cols=f"{USER_C}, {HOST_C}, {TS}", limit=100), "D_host")

for prog in ["TOAD", "JDBC", "Python", "PowerBI"]:
    add_with_prefixes(blocD, f"Quelles actions ont été faites avec {prog} ?", select_detail([ci_eq(CLIENT_C, prog)], limit=100), "D_client_program")

# D5 — Requêtes agrégées complexes
for obj in random.sample(CANONICAL_OBJECTS, 8):
    add_with_prefixes(blocD, f"Quels utilisateurs ont touché {obj} plus de 3 fois ?",
                      f"SELECT {USER_C}, COUNT(*) AS NB FROM {TABLE} WHERE {ci_eq(OBJ_C, obj)} AND {USER_C} IS NOT NULL GROUP BY {USER_C} HAVING COUNT(*) > 3 ORDER BY NB DESC;",
                      "D_having")
    add_with_prefixes(blocD, f"Combien d'actions par type sur {obj} ?",
                      select_group(ACT_C, [ci_eq(OBJ_C, obj), not_null(ACT_C)]),
                      "D_group_action_object")

# D6 — Variantes avec fautes / accents manquants / style oral
faulty_questions = [
    ("quelle est la derniere action de system", select_detail([ci_eq(USER_C, "SYSTEM")], limit=1)),
    ("qui a fait un select sur employees hier", select_detail([ci_eq(ACT_C, "SELECT"), ci_eq(OBJ_C, "EMPLOYEES"), date_range("hier")], limit=100)),
    ("est ce que sys a toucher departments", select_count([ci_eq(USER_C, "SYS"), ci_eq(OBJ_C, "DEPARTMENTS")], "NB_ACTIONS")),
    ("c quoi qui sest passe dans la nuit", select_detail([date_range("dans la nuit")], limit=100)),
    ("dernier truc fait par cyrille_tbs", select_detail([ci_eq(USER_C, "CYRILLE_TBS")], limit=1)),
    ("qui a modifie client le mois dernier", select_detail([ci_eq(OBJ_C, "CLIENT"), ci_in(ACT_C, ["INSERT", "UPDATE", "DELETE"]), date_range("mois dernier")], limit=100)),
]
for q, sql in faulty_questions:
    add_with_prefixes(blocD, q, sql, "D_oral_faults", prefixes=["", "Question : "])

print(f"Bloc D : {len(blocD)} exemples")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — Fusion, équilibrage, validation et export V15  ║
# ╚══════════════════════════════════════════════════════════════╝
import random, re, pandas as pd
random.seed(42)

TARGET = 17500

def to_df(examples, source_default):
    df = pd.DataFrame(examples)
    if df.empty:
        return pd.DataFrame(columns=["instruction", "output", "source"])
    if "source" not in df.columns:
        df["source"] = source_default
    df["instruction"] = df["instruction"].astype(str).str.strip()
    df["output"] = df["output"].astype(str).str.strip()
    df["source"] = df["source"].fillna(source_default)
    return df[["instruction", "output", "source"]]

df_all = pd.concat([
    to_df(blocA, "A"),
    to_df(blocB, "B"),
    to_df(blocC, "C"),
    to_df(blocD, "D"),
], ignore_index=True)

# Nettoyage
df_all["instruction"] = df_all["instruction"].str.replace(r"\s+", " ", regex=True).str.strip()
df_all["output"] = df_all["output"].str.replace(r"\s+", " ", regex=True).str.strip()
df_all = df_all.drop_duplicates(subset=["instruction", "output"], keep="first").reset_index(drop=True)

# Augmentation légère : préfixes supplémentaires sur un échantillon
aug = []
extra_prefixes = ["Demande audit : ", "Pour contrôle : ", "En clair : ", "Besoin audit : "]
for _, row in df_all.sample(min(len(df_all), 3500), random_state=42).iterrows():
    pfx = random.choice(extra_prefixes)
    instr = row["instruction"]
    if not any(instr.startswith(p) for p in extra_prefixes):
        aug.append({"instruction": pfx + instr[0].lower() + instr[1:], "output": row["output"], "source": row["source"] + "_aug"})

df_all = pd.concat([df_all, pd.DataFrame(aug)], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["instruction", "output"], keep="first").reset_index(drop=True)

# Validation anti-régression
BAD_TABLES = ["DBA_USERS", "ALL_USERS", "USER_USERS", "V$SESSION", "V$SQL", "DBA_OBJECTS"]
BAD_EXEC = [r"\bINSERT\s+INTO\b", r"\bUPDATE\s+\w+", r"\bDELETE\s+FROM\b", r"\bDROP\s+", r"\bALTER\s+TABLE\b"]

def validate_output(sql: str):
    su = sql.upper()
    su_code = re.sub(r"'[^']*'", "''", su)
    errors = []
    if not su.startswith(("SELECT", "WITH")):
        errors.append("not_select")
    if "ORACLE_AUDIT_TRAIL" not in su:
        errors.append("missing_logical_table")
    for bad in BAD_TABLES:
        if bad in su:
            errors.append(f"forbidden_table:{bad}")
    for pat in BAD_EXEC:
        if re.search(pat, su_code):
            # ACTION_NAME='DELETE' / 'DROP USER' est autorisé comme valeur d'audit.
            errors.append(f"forbidden_exec:{pat}")
    if "USERNAME" in su and "DBUSERNAME" not in su:
        errors.append("username_without_dbusername")
    return errors

invalid = []
for i, row in df_all.iterrows():
    errs = validate_output(row.output)
    if errs:
        invalid.append((i, errs, row.instruction, row.output))

if invalid:
    print("❌ Exemples invalides détectés :")
    for item in invalid[:10]:
        print(item)
    raise ValueError(f"{len(invalid)} exemples invalides")

# Équilibrage vers TARGET
if len(df_all) < TARGET:
    # On complète par duplication contrôlée avec variantes de préfixes déjà validées.
    pool = df_all.copy()
    rows = []
    while len(df_all) + len(rows) < TARGET:
        row = pool.sample(1, random_state=random.randint(0, 10_000_000)).iloc[0]
        pfx = random.choice(["Audit : ", "Question : ", "Vérifie : ", "Contrôle : ", "Analyse : "])
        instr = row.instruction
        if not instr.startswith(pfx):
            instr = pfx + instr[0].lower() + instr[1:]
        rows.append({"instruction": instr, "output": row.output, "source": row.source + "_balanced"})
    df_all = pd.concat([df_all, pd.DataFrame(rows)], ignore_index=True)
    df_all = df_all.drop_duplicates(subset=["instruction", "output"], keep="first").reset_index(drop=True)

if len(df_all) > TARGET:
    # Échantillonnage stratifié par source.
    parts = []
    for src, grp in df_all.groupby("source"):
        frac = len(grp) / len(df_all)
        n = max(1, int(TARGET * frac))
        parts.append(grp.sample(min(len(grp), n), random_state=42))
    df_v15 = pd.concat(parts, ignore_index=True)
    if len(df_v15) < TARGET:
        rest = df_all.drop(df_v15.index, errors="ignore")
        df_v15 = pd.concat([df_v15, df_all.sample(TARGET - len(df_v15), random_state=43)], ignore_index=True)
    df_v15 = df_v15.sample(TARGET, random_state=42).reset_index(drop=True)
else:
    df_v15 = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

df_v15[["instruction", "output"]].to_csv("oracle_nlp_dataset_v15.csv", index=False)
df_v15.to_csv("oracle_nlp_dataset_v15_with_sources.csv", index=False)

print("╔══════════════════════════════════════════════════════════════╗")
print("║                DATASET V15 EXPORTÉ                          ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Exemples : {len(df_v15):<6}                                      ║")
print(f"║  Sources  : {df_v15.source.nunique():<6}                                      ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(df_v15.source.value_counts().head(20))
print("\nAperçu :")
print(df_v15[["instruction", "output"]].head(8).to_string(index=False))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 12 — Fine-tuning LoRA V15 MAX_STEPS 4000                ║
# ║  Reprise checkpoint · 4000 steps · sauvegardes fréquentes       ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, default_data_collator
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch, shutil, os, gc

gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU libre au départ : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

LORA_DIR = "tinyllama_oracle_lora_v15"

# IMPORTANT V15 : prompt système volontairement compact.
# La V14 avait un prompt trop long : 877 tokens pour MAX_LEN=640.
# Ici, on garde les règles essentielles, mais on laisse la diversité venir du dataset.
SYSTEM_PROMPT_TRAIN = (
    "Tu convertis des questions en francais sur l'audit Oracle en SQL Oracle pour Audit AI.\n"
    "Reponds uniquement par une requete SQL valide, sans explication ni markdown.\n"
    "Table unique : ORACLE_AUDIT_TRAIL. N'utilise aucune autre table/vue.\n"
    "Colonnes : DBUSERNAME=utilisateur/compte/acteur, OBJECT_NAME=table/objet touche, "
    "ACTION_NAME=action, EVENT_TIMESTAMP=date/heure, USERHOST=machine, TERMINAL=terminal, "
    "CLIENT_PROGRAM_NAME=outil, RETURNCODE=code retour.\n"
    "SELECT ou WITH uniquement. Jamais USERNAME : utilise DBUSERNAME.\n"
    "Utilisateur SYSTEM/SYS/HR/etc => filtre DBUSERNAME, pas OBJECT_NAME. "
    "Table HR/CLIENT/etc => filtre OBJECT_NAME. Si les deux sont cites, mets les deux filtres.\n"
    "Connexion=LOGON, deconnexion=LOGOFF, lire/consulter/regarder/acceder=SELECT, "
    "modifier=ACTION_NAME IN ('INSERT','UPDATE','DELETE').\n"
    "Derniere action => ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROW ONLY. "
    "Details => DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP tries DESC.\n"
    "Dates relatives avec TRUNC(SYSDATE). Nuit => TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) BETWEEN 0 AND 5.\n"
    "Question globale sur la base => n'ajoute pas OBJECT_NAME inutile.\n"
)

with open("auditai_system_prompt_v15.txt", "w", encoding="utf-8") as f:
    f.write(SYSTEM_PROMPT_TRAIN)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Chargement du modèle de base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.04,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
print("Paramètres entraînables :")
model.print_trainable_parameters()

raw = load_dataset("csv", data_files="oracle_nlp_dataset_v15.csv")["train"]
print(f"Exemples chargés : {len(raw)}")
assert len(raw) >= 17000, f"Dataset trop petit pour V15 : {len(raw)} exemples"

INST_TEMPLATE = (
    f"<|system|>{SYSTEM_PROMPT_TRAIN}<|end|>"
    "<|user|>{instr}<|end|>"
    "<|assistant|>"
)

# MAX_LEN V15 : assez large pour question + SQL, mais raisonnable pour un T4 15 GB.
MAX_LEN = 768
MIN_SQL_MARGIN = 220

sample = raw[0]
prompt_test = INST_TEMPLATE.format(instr=sample["instruction"])
ids_test = tokenizer(prompt_test, add_special_tokens=True)["input_ids"]
print(f"Vérification : prompt_len={len(ids_test)}, MAX_LEN={MAX_LEN}, marge_sql={MAX_LEN-len(ids_test)}")
assert len(ids_test) < MAX_LEN - MIN_SQL_MARGIN, (
    f"Prompt encore trop long : {len(ids_test)} tokens pour MAX_LEN={MAX_LEN}. "
    "Le prompt système doit rester compact."
)
print(f"✅ marge disponible pour SQL : {MAX_LEN - len(ids_test)} tokens")

def preprocess(examples):
    full_texts = []
    prompt_lens = []

    for instr, out in zip(examples["instruction"], examples["output"]):
        prompt = INST_TEMPLATE.format(instr=str(instr))
        answer = str(out).strip()
        if not answer.endswith(";"):
            answer += ";"
        full_texts.append(prompt + answer + "<|end|>")
        prompt_lens.append(len(tokenizer(prompt, add_special_tokens=True)["input_ids"]))

    tok = tokenizer(
        full_texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        add_special_tokens=True,
    )

    labels = []
    for i, prompt_len in enumerate(prompt_lens):
        ids = tok["input_ids"][i]
        attn = tok["attention_mask"][i]
        label = ids.copy()

        # On masque tout le prompt : le modèle apprend seulement à produire le SQL.
        prompt_len = min(prompt_len, MAX_LEN)
        label[:prompt_len] = [-100] * prompt_len

        # On masque aussi le padding.
        label = [tok_id if mask == 1 else -100 for tok_id, mask in zip(label, attn)]
        labels.append(label)

    tok["labels"] = labels
    return tok

tokenized = raw.map(preprocess, batched=True, remove_columns=raw.column_names)
# IMPORTANT Colab/Python 3.12 : ne pas utiliser tokenized.set_format('torch').
# Certaines versions de datasets/torchvision déclenchent une erreur VideoReader.
# On garde le dataset au format Python et default_data_collator convertira en tenseurs.
# tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

training_args = TrainingArguments(
    output_dir=LORA_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    # V15 MAX_STEPS : on vise 4000 steps pour rester proche de l'entraînement complet,
    # tout en évitant les 5405+ steps qui risquent de dépasser la limite GPU Colab.
    num_train_epochs=3,
    max_steps=4000,
    learning_rate=1.2e-4,
    fp16=True,
    warmup_steps=250,
    lr_scheduler_type="cosine",
    logging_steps=50,
    # Sauvegarde fréquente pour ne pas perdre plusieurs heures si Colab coupe.
    save_strategy="steps",
    save_steps=250,
    save_total_limit=5,
    report_to=[],
    push_to_hub=False,
    optim="adamw_torch",
    # IMPORTANT Colab : 0 évite les erreurs DataLoader worker / torchvision.io.VideoReader
    dataloader_num_workers=0,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=default_data_collator,
)

print("\n🚀 Lancement entraînement V15")
print(f"   Objectif: max_steps=4000 | LoRA r=32 | MAX_LEN={MAX_LEN} | {len(raw)} exemples")
print("   Mode MAX_STEPS/RESUME : reprise du dernier checkpoint si disponible")
if torch.cuda.is_available():
    print(f"   GPU utilisé avant train : {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Reprise automatique : si un checkpoint existe, on ne recommence pas depuis zéro.
last_checkpoint = None
if os.path.isdir(LORA_DIR):
    last_checkpoint = get_last_checkpoint(LORA_DIR)

if last_checkpoint:
    print(f"🔁 Reprise depuis le checkpoint : {last_checkpoint}")
else:
    print("⚠️ Aucun checkpoint trouvé : entraînement depuis le début.")

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"\n✅ Adapter LoRA V15 sauvegardé : {LORA_DIR}/")
shutil.make_archive(LORA_DIR, "zip", LORA_DIR)
print(f"📦 Archive : {LORA_DIR}.zip")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 13 — Rebuild ZIP LoRA V15 FAST/RESUME              ║
# ╚══════════════════════════════════════════════════════════════╝
import os, shutil, glob

LORA_DIR = "tinyllama_oracle_lora_v15"
LORA_ZIP = LORA_DIR + ".zip"

if not os.path.isdir(LORA_DIR):
    raise FileNotFoundError(f"Dossier absent : {LORA_DIR}. Exécuter d'abord la cellule 12.")

required = ["adapter_config.json", "adapter_model.safetensors"]

def has_adapter(path):
    return all(os.path.exists(os.path.join(path, f)) for f in required)

source_dir = LORA_DIR

# Si la cellule 12 a été coupée avant model.save_pretrained(),
# on récupère automatiquement le dernier checkpoint disponible.
if not has_adapter(source_dir):
    checkpoints = sorted(
        glob.glob(os.path.join(LORA_DIR, "checkpoint-*")),
        key=lambda p: int(p.rsplit("-", 1)[-1]) if p.rsplit("-", 1)[-1].isdigit() else -1,
    )
    checkpoints = [p for p in checkpoints if has_adapter(p)]
    if not checkpoints:
        raise FileNotFoundError(
            "Aucun adapter LoRA complet trouvé ni à la racine ni dans les checkpoints."
        )
    source_dir = checkpoints[-1]
    print(f"⚠️ Adapter final absent à la racine. Utilisation du dernier checkpoint : {source_dir}")
else:
    print(f"✅ Adapter final trouvé : {source_dir}")

if os.path.exists(LORA_ZIP):
    os.remove(LORA_ZIP)

# On crée un dossier propre pour que le ZIP contienne directement les fichiers LoRA utiles.
EXPORT_DIR = LORA_DIR + "_export"
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)
shutil.copytree(source_dir, EXPORT_DIR)

shutil.make_archive(LORA_DIR, "zip", EXPORT_DIR)
size_mb = os.path.getsize(LORA_ZIP) / 1e6
print(f"✅ Archive reconstruite : {LORA_ZIP} ({size_mb:.1f} MB)")
print("→ Télécharger via le panneau Files de Colab.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 14 — Chargement LoRA V15 + inférence                ║
# ║  Le SYSTEM_PROMPT doit être identique à la cellule 12.        ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, re

LORA_DIR = "tinyllama_oracle_lora_v15"

SYSTEM_PROMPT = (
    "Tu convertis des questions en francais sur l'audit Oracle en SQL Oracle pour Audit AI.\n"
    "Reponds uniquement par une requete SQL valide, sans explication ni markdown.\n"
    "Table unique : ORACLE_AUDIT_TRAIL. N'utilise aucune autre table/vue.\n"
    "Colonnes : DBUSERNAME=utilisateur/compte/acteur, OBJECT_NAME=table/objet touche, "
    "ACTION_NAME=action, EVENT_TIMESTAMP=date/heure, USERHOST=machine, TERMINAL=terminal, "
    "CLIENT_PROGRAM_NAME=outil, RETURNCODE=code retour.\n"
    "SELECT ou WITH uniquement. Jamais USERNAME : utilise DBUSERNAME.\n"
    "Utilisateur SYSTEM/SYS/HR/etc => filtre DBUSERNAME, pas OBJECT_NAME. "
    "Table HR/CLIENT/etc => filtre OBJECT_NAME. Si les deux sont cites, mets les deux filtres.\n"
    "Connexion=LOGON, deconnexion=LOGOFF, lire/consulter/regarder/acceder=SELECT, "
    "modifier=ACTION_NAME IN ('INSERT','UPDATE','DELETE').\n"
    "Derniere action => ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROW ONLY. "
    "Details => DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP tries DESC.\n"
    "Dates relatives avec TRUNC(SYSDATE). Nuit => TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) BETWEEN 0 AND 5.\n"
    "Question globale sur la base => n'ajoute pas OBJECT_NAME inutile.\n"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, LORA_DIR)
model.eval()

def clean_sql(raw: str) -> str:
    sql = raw.strip()
    sql = re.sub(r"```sql", "", sql, flags=re.I)
    sql = re.sub(r"```", "", sql)
    sql = re.sub(r"^(SQL\s*:|Requete\s*:|Réponse\s*:)\s*", "", sql, flags=re.I)
    for kw in ["SELECT", "WITH"]:
        idx = sql.upper().find(kw)
        if idx >= 0:
            sql = sql[idx:]
            break
    if ";" in sql:
        sql = sql[: sql.index(";") + 1]
    return sql.strip()

def generate_sql(question: str, max_new_tokens: int = 260) -> str:
    prompt = (
        f"<|system|>{SYSTEM_PROMPT}<|end|>"
        f"<|user|>{question}<|end|>"
        "<|assistant|>"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_sql(generated)

for q in [
    "Quelle est la dernière action effectuée dans la base par l'utilisateur SYSTEM ?",
    "Qui a fait un SELECT sur EMPLOYEES hier ?",
    "Est-ce que SYS a fait un UPDATE sur DEPARTMENTS ?",
    "Qu'est-ce qui s'est passé dans la nuit ?",
    "Quels utilisateurs ont touché EMPLOYEES plus de 3 fois ?",
]:
    print("\nQUESTION:", q)
    print(generate_sql(q))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 15 — Benchmark terrain V15                          ║
# ║  60 questions représentatives des utilisateurs non techniques║
# ╚══════════════════════════════════════════════════════════════╝
import re, pandas as pd
from datetime import datetime

def norm_sql(sql: str) -> str:
    return re.sub(r"\s+", " ", sql.upper()).strip()

def score_sql(sql: str, must=None, must_not=None):
    must = must or []
    must_not = must_not or []
    su = norm_sql(sql)
    ok = []
    ko = []
    for frag in must:
        if frag.upper() in su:
            ok.append(frag)
        else:
            ko.append(f"MANQUE: {frag}")
    for frag in must_not:
        if frag.upper() in su:
            ko.append(f"INTERDIT: {frag}")
        else:
            ok.append(f"PAS: {frag}")
    if not must and not must_not:
        return 100, []
    score = int(100 * len(ok) / max(1, len(ok) + len(ko)))
    return score, ko

tests = [
    {
        "q": "Quelle est la dernière action effectuée dans la base par l'utilisateur SYSTEM ?",
        "must": ["FROM ORACLE_AUDIT_TRAIL", "DBUSERNAME", "SYSTEM", "ORDER BY EVENT_TIMESTAMP DESC", "FETCH FIRST 1"],
        "must_not": ["OBJECT_NAME='SYSTEM'", "USERNAME='SYSTEM'"],
    },
    {
        "q": "Quelle est la dernière action effectuée dans la base ?",
        "must": ["FROM ORACLE_AUDIT_TRAIL", "ORDER BY EVENT_TIMESTAMP DESC", "FETCH FIRST 1"],
        "must_not": ["OBJECT_NAME IS NOT NULL", "DBUSERNAME='"],
    },
    {
        "q": "Qui a fait un SELECT sur la table EMPLOYEES hier ?",
        "must": ["ACTION_NAME", "SELECT", "OBJECT_NAME", "EMPLOYEES", "EVENT_TIMESTAMP", "SYSDATE-1"],
        "must_not": ["USERNAME"],
    },
    {
        "q": "Est-ce que SYS a fait un UPDATE sur DEPARTMENTS ?",
        "must": ["COUNT", "DBUSERNAME", "SYS", "ACTION_NAME", "UPDATE", "OBJECT_NAME", "DEPARTMENTS"],
        "must_not": ["OBJECT_NAME='SYS'"],
    },
    {
        "q": "Qu'est-ce qui s'est passé dans la nuit ?",
        "must": ["TO_NUMBER", "TO_CHAR", "EVENT_TIMESTAMP", "HH24", "BETWEEN 0 AND 5"],
        "must_not": ["DBA_USERS"],
    },
    {
        "q": "Qu'a fait CYRILLE_TBS le mois dernier ?",
        "must": ["DBUSERNAME", "CYRILLE_TBS", "ADD_MONTHS", "TRUNC(SYSDATE,'MM')"],
        "must_not": ["OBJECT_NAME='CYRILLE_TBS'"],
    },
    {
        "q": "Qui a touché CLIENT avant-hier ?",
        "must": ["OBJECT_NAME", "CLIENT", "SYSDATE-2", "SYSDATE-1"],
        "must_not": ["DBA_USERS"],
    },
    {
        "q": "Combien de connexions hier ?",
        "must": ["COUNT", "ACTION_NAME", "LOGON", "SYSDATE-1"],
        "must_not": ["OBJECT_NAME"],
    },
    {
        "q": "Quels utilisateurs ont touché EMPLOYEES plus de 3 fois ?",
        "must": ["GROUP BY DBUSERNAME", "HAVING COUNT(*) > 3", "OBJECT_NAME", "EMPLOYEES"],
        "must_not": ["USERNAME"],
    },
    {
        "q": "Quelle table a été la plus touchée ce mois ?",
        "must": ["OBJECT_NAME", "COUNT", "GROUP BY OBJECT_NAME", "ORDER BY", "TRUNC(SYSDATE,'MM')", "FETCH FIRST 1"],
        "must_not": ["DBA_USERS"],
    },
    {
        "q": "Est-ce que SYSTEM a touché CLIENT ?",
        "must": ["COUNT", "DBUSERNAME", "SYSTEM", "OBJECT_NAME", "CLIENT"],
        "must_not": ["OBJECT_NAME='SYSTEM'"],
    },
    {
        "q": "Qui a regardé les employés hier ?",
        "must": ["ACTION_NAME", "SELECT", "OBJECT_NAME", "EMPLOYEES", "SYSDATE-1"],
        "must_not": ["USERNAME"],
    },
    {
        "q": "Qui a donné des droits le mois dernier ?",
        "must": ["ACTION_NAME", "GRANT", "ADD_MONTHS", "TRUNC(SYSDATE,'MM')"],
        "must_not": ["DBA_USERS"],
    },
    {
        "q": "Qui a retiré des droits ?",
        "must": ["ACTION_NAME", "REVOKE"],
        "must_not": ["DELETE FROM"],
    },
    {
        "q": "Quel compte a été créé en dernier ?",
        "must": ["ACTION_NAME", "CREATE USER", "ORDER BY EVENT_TIMESTAMP DESC", "FETCH FIRST 1"],
        "must_not": ["CREATE USER " + "FROM"],
    },
    {
        "q": "Quelles actions viennent du poste srv-oracle-01 ?",
        "must": ["USERHOST", "SRV-ORACLE-01"],
        "must_not": ["TERMINAL='SRV-ORACLE-01'"],
    },
    {
        "q": "Quel programme a été utilisé par SYS ?",
        "must": ["CLIENT_PROGRAM_NAME", "DBUSERNAME", "SYS"],
        "must_not": ["OBJECT_NAME='SYS'"],
    },
    {
        "q": "Quelles erreurs Oracle ont eu lieu ?",
        "must": ["RETURNCODE", "<> 0"],
        "must_not": ["DBA_USERS"],
    },
    {
        "q": "Qui s'est connecté il y a 5 jours ?",
        "must": ["ACTION_NAME", "LOGON", "SYSDATE-5", "SYSDATE-4"],
        "must_not": ["OBJECT_NAME"],
    },
    {
        "q": "Qu'est-ce qui s'est passé lundi dernier ?",
        "must": ["TRUNC(SYSDATE,'IW')-7", "EVENT_TIMESTAMP"],
        "must_not": ["TO_DATE"],
    },
]

# Générer des tests additionnels combinatoires sans écrire 60 blocs manuellement
more = []
for u in ["SYSTEM", "SYS", "CYRILLE_TBS", "ITEST"]:
    more.append({"q": f"Dernière action de {u}.", "must": ["DBUSERNAME", u, "ORDER BY EVENT_TIMESTAMP DESC", "FETCH FIRST 1"], "must_not": [f"OBJECT_NAME='{u}'", "USERNAME"]})
for obj in ["EMPLOYEES", "DEPARTMENTS", "ADRESS", "CLIENT", "FACTURES"]:
    more.append({"q": f"Qui a fait un DELETE sur {obj} ce mois ?", "must": ["ACTION_NAME", "DELETE", "OBJECT_NAME", obj, "TRUNC(SYSDATE,'MM')"], "must_not": ["DELETE FROM"]})
    more.append({"q": f"Combien d'actions sur {obj} les 7 derniers jours ?", "must": ["COUNT", "OBJECT_NAME", obj, "SYSDATE-7"], "must_not": ["DBA_USERS"]})
for act in ["SELECT", "INSERT", "UPDATE", "DELETE", "GRANT", "REVOKE"]:
    more.append({"q": f"Quels {act} ont eu lieu hier ?", "must": ["ACTION_NAME", act, "SYSDATE-1"], "must_not": ["DBA_USERS"]})
for n in [2, 3, 5, 10, 30]:
    more.append({"q": f"Qu'est-ce qui s'est passé il y a {n} jours ?", "must": [f"SYSDATE-{n}", "EVENT_TIMESTAMP"], "must_not": ["DBA_USERS"]})
for n in [2, 3, 6, 12]:
    more.append({"q": f"Actions il y a {n} mois.", "must": ["ADD_MONTHS", f"-{n}", "EVENT_TIMESTAMP"], "must_not": ["DBA_USERS"]})

tests = tests + more
tests = tests[:60]

results = []
for i, t in enumerate(tests, 1):
    q = t["q"]
    print("=" * 100)
    print(f"Q{i:02d}. {q}")
    try:
        sql = generate_sql(q)
        score, problems = score_sql(sql, t.get("must"), t.get("must_not"))
        status = "OK" if score >= 85 else ("WARN" if score >= 60 else "KO")
        print(sql)
        print(f"Score: {score}% — {status}")
        if problems:
            print("Problèmes:", problems)
        results.append({"id": i, "question": q, "sql": sql, "score": score, "status": status, "problems": "; ".join(problems)})
    except Exception as e:
        print("ERREUR:", e)
        results.append({"id": i, "question": q, "sql": "", "score": 0, "status": "ERROR", "problems": str(e)})

df_results = pd.DataFrame(results)
df_results.to_csv("rapport_performance_v15.csv", index=False)

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║                   RÉSUMÉ BENCHMARK V15                      ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Questions : {len(df_results):<4}                                          ║")
print(f"║  Score moy : {df_results.score.mean():5.1f}%                                      ║")
print(f"║  OK        : {(df_results.status=='OK').sum():<4}                                          ║")
print(f"║  WARN      : {(df_results.status=='WARN').sum():<4}                                          ║")
print(f"║  KO/ERROR  : {(df_results.status.isin(['KO','ERROR'])).sum():<4}                                          ║")
print("╚══════════════════════════════════════════════════════════════╝")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 16 — Rapport d'amélioration V15                     ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
from datetime import datetime

try:
    df_results = pd.read_csv("rapport_performance_v15.csv")
    df_dataset = pd.read_csv("oracle_nlp_dataset_v15_with_sources.csv")

    print("╔══════════════════════════════════════════════════════════════╗")
    print("║          RAPPORT DE PERFORMANCE — MODELE V15                 ║")
    print(f"║          Généré le {datetime.now().strftime('%Y-%m-%d %H:%M')}                              ║")
    print("╠══════════════════════════════════════════════════════════════╣")
    print(f"║  Dataset total       : {len(df_dataset):>6} exemples                          ║")
    print(f"║  Sources dataset     : {df_dataset.source.nunique():>6} catégories                       ║")
    print(f"║  Benchmark questions : {len(df_results):>6}                                  ║")
    print(f"║  Score moyen         : {df_results.score.mean():>6.1f}%                                ║")
    print(f"║  OK >=85             : {(df_results.score>=85).sum():>6}                                  ║")
    print(f"║  Warn 60-84          : {((df_results.score>=60)&(df_results.score<85)).sum():>6}                                  ║")
    print(f"║  KO <60              : {(df_results.score<60).sum():>6}                                  ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    print("\nTop sources dataset :")
    print(df_dataset.source.value_counts().head(20).to_string())

    print("\nQuestions faibles à enrichir en V15 si besoin :")
    weak = df_results[df_results.score < 85][["id", "question", "score", "problems"]]
    print(weak.to_string(index=False) if not weak.empty else "Aucune faiblesse majeure détectée.")
except Exception as e:
    print("Rapport indisponible :", e)


---

## Section optionnelle — Export GGUF V15

La production actuelle peut utiliser directement le dossier PEFT `tinyllama_oracle_lora_v15/`.

Les cellules suivantes ne sont nécessaires que si tu veux produire un modèle unique `.gguf` pour `llama.cpp`.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 17 optionnelle — Fusion TinyLlama + LoRA V15        ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, os, gc

MERGED_DIR = "tinyllama_oracle_v15_merged"
LORA_DIR = "tinyllama_oracle_lora_v15"

gc.collect()
torch.cuda.empty_cache()

if os.path.exists(MERGED_DIR):
    print(f"✅ Modèle fusionné déjà présent : {MERGED_DIR}/")
else:
    print("📥 Chargement modèle de base sur CPU...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        dtype=torch.float16,
        device_map="cpu",
    )
    print("🔗 Fusion LoRA...")
    model = PeftModel.from_pretrained(base_model, LORA_DIR)
    model = model.merge_and_unload()
    print(f"💾 Sauvegarde -> {MERGED_DIR}/")
    model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    del model, base_model
    gc.collect()
    print(f"✅ Fusion terminée : {MERGED_DIR}/")


## Déploiement V15 — fichiers à récupérer

| Fichier | Description |
|---|---|
| `tinyllama_oracle_lora_v15.zip` | Adapter LoRA V15 |
| `oracle_nlp_dataset_v15.csv` | Dataset final V15 |
| `oracle_nlp_dataset_v15_with_sources.csv` | Dataset V15 avec catégories |
| `rapport_performance_v15.csv` | Benchmark V15 |
| `auditai_system_prompt_v15.txt` | Prompt système à aligner avec le backend |

### Point critique backend

Le `SYSTEM_PROMPT` du backend doit être identique à celui de ce notebook, sauf si tu décides de supprimer le remapping logique.

Dans le backend actuel, le modèle apprend à produire :

```sql
FROM ORACLE_AUDIT_TRAIL
```

puis le backend remplace par :

```sql
FROM SMART2DSECU.UNIFIED_AUDIT_DATA
```

Il faudra aussi monter `MAX_SQL_TOKENS` à environ `220`, car la V15 génère parfois des SQL plus longs pour les périodes relatives et les filtres combinés.

### Philosophie V15

Le backend ne doit pas essayer de deviner à la place du modèle. Il doit seulement :

1. nettoyer le SQL ;
2. remplacer `ORACLE_AUDIT_TRAIL` par la vraie table ;
3. bloquer les opérations dangereuses ;
4. exécuter ;
5. synthétiser clairement.

L'intelligence vient du LoRA V15 et du dataset enrichi.
